In [39]:
import os
import numpy as np
import torch
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch # Đảm bảo đã import torch để kiểm tra kiểu dữ liệu
import cv2
import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn, FasterRCNN_MobileNet_V3_Large_320_FPN_Weights
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.autonotebook import tqdm


In [40]:
import os
import cv2
import torch
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset
import numpy as np

class ISICDataset(Dataset):
    def __init__(self, root, subset, transforms=None):
        self.root = os.path.join(root, subset)
        self.transforms = transforms
        
        # 1. Lấy danh sách ảnh và sắp xếp để đảm bảo tính nhất quán
        self.imgs = sorted([f for f in os.listdir(self.root) 
                           if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        
        # 2. Tự động tạo Label Map từ dữ liệu thực tế
        self.label_map = self._create_label_map()
        
    def _create_label_map(self):
        unique_labels = set()
        xml_files = [f for f in os.listdir(self.root) if f.lower().endswith('.xml')]
        for xml_file in xml_files:
            tree = ET.parse(os.path.join(self.root, xml_file))
            root = tree.getroot()
            for obj in root.findall('object'):
                unique_labels.add(obj.find('name').text)
        
        # ID 0 dành cho background, các bệnh từ 1-9
        mapping = {name: i + 1 for i, name in enumerate(sorted(list(unique_labels)))}
        mapping["background"] = 0
        return mapping

    def __getitem__(self, idx):
        # Đường dẫn ảnh và XML tương ứng
        img_path = os.path.join(self.root, self.imgs[idx])
        xml_path = os.path.join(self.root, os.path.splitext(self.imgs[idx])[0] + ".xml")

        # Đọc ảnh bằng OpenCV và chuyển sang RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            labels.append(self.label_map[name])
            
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)
            
            # Kiểm tra tính hợp lệ của box để tránh lỗi tọa độ âm hoặc bằng không
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])

        # Áp dụng Albumentations (nếu có)
        if self.transforms:
            transformed = self.transforms(image=img, bboxes=boxes, labels=labels)
            img = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']

        # --- PHẦN SỬA LỖI QUAN TRỌNG CHO FASTER R-CNN ---
        
        # Đảm bảo boxes luôn có dạng [N, 4], kể cả khi rỗng
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Tính toán diện tích (area) trực tiếp từ tensor boxes
        if boxes.shape[0] > 0:
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        else:
            area = torch.tensor([0], dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64)
        }

        # Chuyển đổi ảnh sang tensor float32 và chuẩn hóa về [0, 1]
        if not isinstance(img, torch.Tensor):
            img = torch.from_numpy(img).permute(2, 0, 1).to(torch.float32) / 255.0
        elif img.dtype != torch.float32:
            img = img.to(torch.float32) / 255.0

        return img, target

    def __len__(self):
        return len(self.imgs)

# Hàm ghép nối Batch tùy chỉnh cho Object Detection
def collate_fn(batch):
    return tuple(zip(*batch))

In [41]:

# Import class của bạn (Giả sử file đặt tên là dataset.py)
# from dataset import ISICDataset 

# --- CODE HUẤN LUYỆN ---

def collate_fn(batch):
    return tuple(zip(*batch))

def get_primary_label(xml_path):
    """Lấy nhãn đầu tiên trong file XML để tính trọng số sampler"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    obj = root.find('object')
    return obj.find('name').text if obj is not None else "__empty__"

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Cấu hình đường dẫn
    BASE_DIR = r"D:\xu_li_anh\btl\data"
    OUTPUT_DIR = r"D:\xu_li_anh\btl\checkpoin"
    LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
    MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
    os.makedirs(MODEL_DIR, exist_ok=True)

    # 2. Định nghĩa Transforms (Albumentations)
    # Lưu ý: ISICDataset của bạn đã có dòng img / 255.0, 
    # nên ta không dùng A.Normalize ở đây để tránh bị chia 2 lần.
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Rotate(limit=30, p=0.3),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    val_transform = A.Compose([
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    # 3. Khởi tạo Dataset
    train_ds = ISICDataset(root=BASE_DIR, subset='train', transforms=train_transform)
    valid_ds = ISICDataset(root=BASE_DIR, subset='valid', transforms=val_transform)

    # Lấy thông tin class từ Dataset vừa khởi tạo
    label_map = train_ds.label_map
    num_classes = len(label_map) # Đã bao gồm background (ID 0)
    print(f"Detected Classes: {label_map}")

    # 4. Tính toán WeightedRandomSampler để cân bằng dữ liệu
    print("--- Đang thống kê nhãn để cân bằng dữ liệu tập Train ---")
    train_xml_paths = [os.path.join(train_ds.root, os.path.splitext(f)[0] + ".xml") for f in train_ds.imgs]
    primary_labels = [get_primary_label(p) for p in train_xml_paths]
    
    class_counts = Counter(primary_labels)
    # Trọng số nghịch đảo (dùng căn bậc 2 để tránh oversampling quá mức các lớp hiếm)
    class_weights = {cls: 1.0 / np.sqrt(count) for cls, count in class_counts.items()}
    sample_weights = [class_weights[cls] for cls in primary_labels]

    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    print(f"Số lượng mẫu mỗi lớp: {dict(class_counts)}")

    # 5. Khởi tạo Dataloader
    train_loader = DataLoader(train_ds, batch_size=4, sampler=train_sampler, collate_fn=collate_fn, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

    # 6. Khởi tạo Mô hình (MobileNetV3 cho tốc độ nhanh)
    model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT)
    in_channels = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_channels, num_classes)
    model.to(device)

    # 7. Cấu hình Huấn luyện
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.0005)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    writer = SummaryWriter(log_dir=LOG_DIR)
    map_metric = MeanAveragePrecision(iou_type="bbox")
    
    NUM_EPOCHS = 50
    best_map = 0.0

    for epoch in range(NUM_EPOCHS):
        # --- TRAINING PHASE ---
        model.train()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", colour="green")
        epoch_losses = []
        
        for images, targets in train_pbar:
            images = [img.to(device, dtype=torch.float32) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            epoch_losses.append(losses.item())
            train_pbar.set_postfix({"loss": f"{np.mean(epoch_losses):.4f}"})

        avg_loss = np.mean(epoch_losses)
        writer.add_scalar("Loss/Train", avg_loss, epoch)
        lr_scheduler.step()

        # --- EVALUATION PHASE ---
        model.eval()
        map_metric.reset()
        val_pbar = tqdm(valid_loader, desc=f"Epoch {epoch+1} [Valid]", leave=False)
        
        with torch.no_grad():
            for images, targets in val_pbar:
                images = [img.to(device, dtype=torch.float32) for img in images]
                outputs = model(images)
                
                # Chuyển output về CPU để tính metric
                preds = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
                gts = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
                map_metric.update(preds, gts)
        
        results = map_metric.compute()
        current_map = results['map'].item()
        writer.add_scalar("mAP/Val", current_map, epoch)
        
        print(f"Epoch {epoch+1} Summary: Loss: {avg_loss:.4f} | mAP@0.5:0.95: {current_map:.4f}")

        # Lưu model tốt nhất dựa trên mAP
        if current_map > best_map:
            best_map = current_map
            save_path = os.path.join(MODEL_DIR, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'map': best_map,
                'label_map': label_map
            }, save_path)
            print(f"Saved new best model with mAP: {best_map:.4f}")

    writer.close()
    print("Huấn luyện hoàn tất!")

Detected Classes: {'actinic keratosis': 1, 'basal cell carcinoma': 2, 'dermatofibroma': 3, 'melanoma': 4, 'nevus': 5, 'pigmented benign keratosis': 6, 'seborrheic keratosis': 7, 'squamous cell carcinoma': 8, 'vascular lesion': 9, 'background': 0}
--- Đang thống kê nhãn để cân bằng dữ liệu tập Train ---
Số lượng mẫu mỗi lớp: {'nevus': 80, 'melanoma': 67, 'seborrheic keratosis': 51, 'vascular lesion': 90, 'dermatofibroma': 60, 'actinic keratosis': 79, '__empty__': 3, 'squamous cell carcinoma': 93, 'pigmented benign keratosis': 112, 'basal cell carcinoma': 78}


Epoch 1/50 [Train]: 100%|██████████| 179/179 [00:28<00:00,  6.28it/s, loss=2.4489]


Epoch 1 Summary: Loss: 2.4489 | mAP@0.5:0.95: 0.1317
Saved new best model with mAP: 0.1317


Epoch 2/50 [Train]: 100%|██████████| 179/179 [01:00<00:00,  2.98it/s, loss=2.4204]


Epoch 2 Summary: Loss: 2.4204 | mAP@0.5:0.95: 0.1879
Saved new best model with mAP: 0.1879


...

Epoch 48 Summary: Loss: 0.7506 | mAP@0.5:0.95: 0.3542


Epoch 49/50 [Train]: 100%|██████████| 179/179 [00:25<00:00,  6.89it/s, loss=0.6756]


Epoch 49 Summary: Loss: 0.6756 | mAP@0.5:0.95: 0.3607


Epoch 50/50 [Train]: 100%|██████████| 179/179 [00:21<00:00,  8.31it/s, loss=0.6534]
                                                                 

Epoch 50 Summary: Loss: 0.6534 | mAP@0.5:0.95: 0.3684
Huấn luyện hoàn tất!
